
# Análise final da ablação — RAG vs. Sem RAG

Este notebook **não gera novos relatórios** e **não altera os frameworks**. Ele apenas:

1. recebe os dois arquivos ZIP já produzidos;
2. valida o pareamento dos **24 casos**;
3. associa os mesmos relatórios técnicos de referência às duas condições;
4. calcula métricas por caso;
5. executa análises pareadas globais, por modelo e por classe;
6. exporta tabelas, gráficos e um ZIP final.

## Desenho fixo

- 4 modelos;
- 6 casos/classes por modelo;
- 24 pares;
- condição `with_rag`;
- condição `without_rag`;
- chave de pareamento: `model_key + sample_id`.

As métricas principais são calculadas para **todos os 24 pares**, sem excluir automaticamente saídas ruins ou inválidas. Uma análise de sensibilidade com relatórios válidos é produzida separadamente quando as colunas de auditoria permitirem.


## 1. Instalação das dependências

In [ ]:

!pip -q install sentence-transformers rouge-score bert-score scipy statsmodels openpyxl


## 2. Importações e configuração

In [ ]:

from pathlib import Path
from google.colab import files
from IPython.display import display

import hashlib
import json
import math
import os
import re
import shutil
import zipfile
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from rouge_score import rouge_scorer
from bert_score import score as bert_score

warnings.filterwarnings("ignore")

OUTPUT_DIR = Path("/content/ablation_analysis_final")
INPUT_DIR = OUTPUT_DIR / "inputs"
FIG_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR / "tables"

for directory in [OUTPUT_DIR, INPUT_DIR, FIG_DIR, TABLE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

EXPECTED_MODELS = {"mistral", "qwen", "tinyllama", "zephyr"}
EXPECTED_PAIRS = 24
RANDOM_SEED = 42
N_BOOTSTRAP = 10000
N_PERMUTATIONS = 10000
TIE_TOLERANCE = 1e-12

np.random.seed(RANDOM_SEED)

print("Diretório de saída:", OUTPUT_DIR)
print("GPU disponível:", torch.cuda.is_available())


## 3. Upload dos dois ZIPs

In [ ]:

print("Envie os dois arquivos ZIP:")
print("1) results_slm_rag_final_v4.zip")
print("2) without_rag_results.zip ou without_rag_results(1).zip")

uploaded = files.upload()

zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
if len(zip_names) != 2:
    raise ValueError(
        f"Esperados exatamente 2 ZIPs; recebidos {len(zip_names)}: {zip_names}"
    )

for name in zip_names:
    target = INPUT_DIR / Path(name).name
    target.write_bytes(uploaded[name])

print("ZIPs recebidos:")
for path in sorted(INPUT_DIR.glob("*.zip")):
    print("-", path.name, f"({path.stat().st_size / 1024:.1f} KB)")


## 4. Identificação e leitura automática dos CSVs consolidados

In [ ]:

def list_zip_members(zip_path: Path):
    with zipfile.ZipFile(zip_path) as zf:
        return zf.namelist()


def find_member(zip_path: Path, preferred_names):
    members = list_zip_members(zip_path)
    basename_map = {Path(m).name: m for m in members}

    for preferred in preferred_names:
        if preferred in basename_map:
            return basename_map[preferred]

    csv_members = [m for m in members if m.lower().endswith(".csv")]
    for member in csv_members:
        lower = Path(member).name.lower()
        if "all_models" in lower or "all_generations" in lower:
            return member

    raise FileNotFoundError(
        f"CSV consolidado não encontrado em {zip_path.name}. Conteúdo: {members}"
    )


def read_csv_from_zip(zip_path: Path, member: str) -> pd.DataFrame:
    with zipfile.ZipFile(zip_path) as zf:
        with zf.open(member) as file_obj:
            return pd.read_csv(file_obj)


rag_zip = None
no_rag_zip = None

for zip_path in INPUT_DIR.glob("*.zip"):
    members_lower = " ".join(list_zip_members(zip_path)).lower()
    if "slm_rag_results_all_models" in members_lower:
        rag_zip = zip_path
    elif "all_generations_without_rag" in members_lower:
        no_rag_zip = zip_path

if rag_zip is None or no_rag_zip is None:
    raise RuntimeError(
        "Não foi possível identificar automaticamente os ZIPs com e sem RAG."
    )

rag_member = find_member(
    rag_zip,
    ["slm_rag_results_all_models_final_v4.csv"],
)
no_rag_member = find_member(
    no_rag_zip,
    ["all_generations_without_rag.csv"],
)

rag_raw = read_csv_from_zip(rag_zip, rag_member)
no_rag_raw = read_csv_from_zip(no_rag_zip, no_rag_member)

print("Com RAG:", rag_zip.name, "->", rag_member, rag_raw.shape)
print("Sem RAG:", no_rag_zip.name, "->", no_rag_member, no_rag_raw.shape)

display(rag_raw.head(2))
display(no_rag_raw.head(2))


## 5. Padronização das colunas e das classes

In [ ]:

CLASS_ALIASES = {
    "bearing_failure": "Bearing_Failure",
    "bearing failure": "Bearing_Failure",
    "blocked_rotor": "Blocked_Rotor",
    "blocked rotor": "Blocked_Rotor",
    "lack_of_phase": "Lack_of_Phase",
    "lack of phase": "Lack_of_Phase",
    "phase_loss": "Lack_of_Phase",
    "phase loss": "Lack_of_Phase",
    "normal": "Normal_Operation",
    "normal_operation": "Normal_Operation",
    "normal operation": "Normal_Operation",
    "overheating": "Overheating",
    "ventilation_defect": "Ventilation_Defect",
    "ventilation defect": "Ventilation_Defect",
}


def canonical_class(value):
    text = str(value).strip()
    key = re.sub(r"[-]+", "_", text.lower())
    key = re.sub(r"\s+", " ", key)
    return CLASS_ALIASES.get(key, CLASS_ALIASES.get(key.replace(" ", "_"), text.replace(" ", "_")))


def standardize_condition(df: pd.DataFrame, condition: str) -> pd.DataFrame:
    out = df.copy()

    required = ["sample_id", "model_key", "report"]
    missing = [c for c in required if c not in out.columns]
    if missing:
        raise ValueError(f"{condition}: colunas obrigatórias ausentes: {missing}")

    if "predicted_class" not in out.columns:
        if "true_class" in out.columns:
            out["predicted_class"] = out["true_class"]
        else:
            raise ValueError(f"{condition}: predicted_class/true_class ausentes.")

    out["sample_id"] = out["sample_id"].astype(str).str.strip()
    out["model_key"] = out["model_key"].astype(str).str.lower().str.strip()
    out["canonical_class"] = out["predicted_class"].map(canonical_class)
    out["report"] = out["report"].fillna("").astype(str).str.strip()
    out["condition"] = condition
    out["pair_id"] = out["model_key"] + "::" + out["sample_id"]

    if "valid_report" not in out.columns:
        out["valid_report"] = out["report"].str.len().ge(120)

    out["valid_report"] = (
        out["valid_report"]
        .astype(str)
        .str.lower()
        .map({"true": True, "false": False, "1": True, "0": False})
        .fillna(out["report"].str.len().ge(120))
        .astype(bool)
    )

    return out


rag = standardize_condition(rag_raw, "with_rag")
no_rag = standardize_condition(no_rag_raw, "without_rag")

print("Modelos com RAG:", sorted(rag["model_key"].unique()))
print("Modelos sem RAG:", sorted(no_rag["model_key"].unique()))
print("Classes com RAG:", sorted(rag["canonical_class"].unique()))
print("Classes sem RAG:", sorted(no_rag["canonical_class"].unique()))


## 6. Validação estrutural e pareamento dos 24 casos

In [ ]:

validation_rows = []

def add_check(name, passed, detail):
    validation_rows.append({
        "check": name,
        "passed": bool(passed),
        "detail": str(detail),
    })


add_check("24 registros com RAG", len(rag) == EXPECTED_PAIRS, len(rag))
add_check("24 registros sem RAG", len(no_rag) == EXPECTED_PAIRS, len(no_rag))
add_check(
    "Modelos esperados com RAG",
    set(rag["model_key"]) == EXPECTED_MODELS,
    sorted(set(rag["model_key"])),
)
add_check(
    "Modelos esperados sem RAG",
    set(no_rag["model_key"]) == EXPECTED_MODELS,
    sorted(set(no_rag["model_key"])),
)
add_check(
    "Sem duplicatas com RAG",
    not rag["pair_id"].duplicated().any(),
    rag.loc[rag["pair_id"].duplicated(), "pair_id"].tolist(),
)
add_check(
    "Sem duplicatas sem RAG",
    not no_rag["pair_id"].duplicated().any(),
    no_rag.loc[no_rag["pair_id"].duplicated(), "pair_id"].tolist(),
)

rag_keys = set(rag["pair_id"])
no_rag_keys = set(no_rag["pair_id"])

add_check(
    "Chaves de pareamento idênticas",
    rag_keys == no_rag_keys,
    {
        "somente_com_rag": sorted(rag_keys - no_rag_keys),
        "somente_sem_rag": sorted(no_rag_keys - rag_keys),
    },
)

counts_rag = rag.groupby("model_key").size().to_dict()
counts_no = no_rag.groupby("model_key").size().to_dict()

add_check(
    "6 casos por modelo com RAG",
    all(counts_rag.get(m) == 6 for m in EXPECTED_MODELS),
    counts_rag,
)
add_check(
    "6 casos por modelo sem RAG",
    all(counts_no.get(m) == 6 for m in EXPECTED_MODELS),
    counts_no,
)

validation_df = pd.DataFrame(validation_rows)
display(validation_df)

validation_df.to_csv(
    TABLE_DIR / "structural_validation.csv",
    index=False,
    encoding="utf-8",
)

if not validation_df["passed"].all():
    failed = validation_df.loc[~validation_df["passed"]]
    raise RuntimeError(
        "A validação estrutural falhou. Corrija os dados antes de calcular métricas.\n"
        + failed.to_string(index=False)
    )

paired = rag.merge(
    no_rag,
    on=["pair_id", "sample_id", "model_key"],
    how="inner",
    suffixes=("_rag", "_no_rag"),
    validate="one_to_one",
)

class_match = paired["canonical_class_rag"] == paired["canonical_class_no_rag"]
add_check(
    "Classes iguais dentro de cada par",
    class_match.all(),
    paired.loc[
        ~class_match,
        ["pair_id", "canonical_class_rag", "canonical_class_no_rag"]
    ].to_dict("records"),
)

if not class_match.all():
    raise RuntimeError("Há classes diferentes dentro de pares correspondentes.")

paired["canonical_class"] = paired["canonical_class_rag"]

print(f"Pareamento concluído: {len(paired)} pares.")
display(
    paired[
        [
            "pair_id",
            "model_key",
            "sample_id",
            "canonical_class",
            "valid_report_rag",
            "valid_report_no_rag",
        ]
    ].sort_values(["model_key", "sample_id"])
)



## 7. Relatórios técnicos de referência

Os seis relatórios abaixo são os mesmos relatórios técnicos de referência já empregados nas avaliações anteriores do experimento. Eles são associados **por classe** e usados igualmente nas duas condições.


In [ ]:

REFERENCE_REPORTS = {
    "Bearing_Failure": """
Bearing failure is indicated by localized temperature increase near the bearing housing, often associated with increased friction, lubrication degradation, contamination, misalignment, excessive load, or rolling element wear. Evidence may include vibration, acoustic noise, thermal asymmetry between drive-end and non-drive-end bearings, and progressive mechanical degradation. Maintenance should include inspection of bearing temperature, vibration and noise, verification of lubricant condition and quantity, checking seals and contamination, evaluating shaft and coupling alignment, and replacing the bearing if wear or damage is confirmed.
""".strip(),

    "Blocked_Rotor": """
Blocked rotor is a critical mechanical or electromechanical condition in which the rotor or driven shaft cannot rotate freely. It is typically associated with very high starting current, rapid temperature rise, failure to reach rated speed, abnormal noise, mechanical obstruction, bearing seizure, coupling locking, or excessive load torque. Maintenance should begin with immediate de-energization and lockout, followed by inspection of the shaft, bearings, coupling, fan and driven equipment. Repeated starting attempts must be avoided until the obstruction or mechanical cause is corrected.
""".strip(),

    "Lack_of_Phase": """
Phase loss is an electrical supply fault in which one phase of a three-phase induction motor is interrupted or unavailable. It produces an unbalanced rotating magnetic field, current redistribution in the remaining phases, torque reduction, vibration, thermal imbalance and rapid winding heating. Evidence may include missing voltage in one phase, unbalanced phase currents, increased current in the remaining phases, humming, difficulty starting and protection relay activation. Maintenance actions should include measuring line-to-line voltages and phase currents, inspecting fuses, breakers, contactors, terminals and cables, and verifying phase-failure protection before restarting.
""".strip(),

    "Normal_Operation": """
Normal operation indicates that the motor is operating within expected electrical, thermal, mechanical and environmental limits. No abnormal heating pattern, excessive vibration, phase imbalance or mechanical restriction is detected. Expected evidence includes stable frame temperature, no abnormal localized hot spots, balanced phase currents, acceptable vibration and noise, smooth shaft rotation and stable performance. Maintenance should continue scheduled preventive routines, monitor temperature, current and vibration trends, keep ventilation clean, tighten electrical connections safely and lubricate bearings according to manufacturer recommendations.
""".strip(),

    "Overheating": """
Overheating is an abnormal thermal condition in which the motor operates above its expected temperature for the applied load, ambient condition, duty cycle and cooling configuration. It may result from overload, voltage imbalance, repeated starts, high current, winding defects, poor power quality, mechanical friction, blocked ventilation, dirty cooling channels, fan problems or high ambient temperature. Maintenance should verify load, current, voltage balance, ventilation paths, fan operation, cleanliness, thermal protection settings and insulation condition if the fault persists.
""".strip(),

    "Ventilation_Defect": """
Ventilation defect occurs when the cooling system cannot adequately remove heat from the motor. It is caused by blocked air inlets or outlets, damaged or loose fan, dust accumulation, contaminated cooling fins, insufficient installation clearance, foreign material in ducts, high ambient temperature or inappropriate cooling method. The thermal pattern may resemble overheating, but the cause is specifically ineffective heat dissipation. Maintenance should clean fan covers, air passages, cooling fins and ducts, inspect fan integrity and rotation, verify installation clearance and compare temperature before and after airflow restoration.
""".strip(),
}

missing_refs = sorted(set(paired["canonical_class"]) - set(REFERENCE_REPORTS))
if missing_refs:
    raise ValueError(f"Referências ausentes para: {missing_refs}")

paired["reference_report"] = paired["canonical_class"].map(REFERENCE_REPORTS)

reference_df = pd.DataFrame(
    [
        {"canonical_class": key, "reference_report": value}
        for key, value in REFERENCE_REPORTS.items()
    ]
)
reference_df.to_csv(
    TABLE_DIR / "reference_reports_used.csv",
    index=False,
    encoding="utf-8",
)

display(reference_df[["canonical_class"]])


## 8. Cálculo de Cosine, ROUGE-L e BERTScore

In [ ]:

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

embedding_model = SentenceTransformer(
    "sentence-transformers/all-mpnet-base-v2",
    device=DEVICE,
)
rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)


def encode_normalized(texts):
    return embedding_model.encode(
        [str(x) for x in texts],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
        batch_size=16,
    )


reports_rag = paired["report_rag"].astype(str).tolist()
reports_no = paired["report_no_rag"].astype(str).tolist()
references = paired["reference_report"].astype(str).tolist()

emb_all = encode_normalized(reports_rag + reports_no + references)
n = len(paired)

emb_rag = emb_all[:n]
emb_no = emb_all[n:2*n]
emb_ref = emb_all[2*n:3*n]

paired["cosine_rag"] = np.sum(emb_rag * emb_ref, axis=1)
paired["cosine_no_rag"] = np.sum(emb_no * emb_ref, axis=1)
paired["cosine_delta"] = paired["cosine_rag"] - paired["cosine_no_rag"]

paired["rag_vs_no_rag_cosine"] = np.sum(emb_rag * emb_no, axis=1)

paired["rougeL_rag"] = [
    rouge.score(ref, pred)["rougeL"].fmeasure
    for pred, ref in zip(reports_rag, references)
]
paired["rougeL_no_rag"] = [
    rouge.score(ref, pred)["rougeL"].fmeasure
    for pred, ref in zip(reports_no, references)
]
paired["rougeL_delta"] = paired["rougeL_rag"] - paired["rougeL_no_rag"]

print("Calculando BERTScore para a condição com RAG...")
_, _, bert_rag = bert_score(
    reports_rag,
    references,
    lang="en",
    model_type="microsoft/deberta-xlarge-mnli",
    device=DEVICE,
    verbose=True,
    rescale_with_baseline=False,
)

print("Calculando BERTScore para a condição sem RAG...")
_, _, bert_no = bert_score(
    reports_no,
    references,
    lang="en",
    model_type="microsoft/deberta-xlarge-mnli",
    device=DEVICE,
    verbose=True,
    rescale_with_baseline=False,
)

paired["bertscore_rag"] = bert_rag.cpu().numpy()
paired["bertscore_no_rag"] = bert_no.cpu().numpy()
paired["bertscore_delta"] = (
    paired["bertscore_rag"] - paired["bertscore_no_rag"]
)

metric_cols = [
    "pair_id", "model_key", "sample_id", "canonical_class",
    "valid_report_rag", "valid_report_no_rag",
    "cosine_rag", "cosine_no_rag", "cosine_delta",
    "rougeL_rag", "rougeL_no_rag", "rougeL_delta",
    "bertscore_rag", "bertscore_no_rag", "bertscore_delta",
    "rag_vs_no_rag_cosine",
]

display(paired[metric_cols].sort_values(["model_key", "sample_id"]))

paired.to_csv(
    TABLE_DIR / "paired_rag_vs_no_rag_with_metrics.csv",
    index=False,
    encoding="utf-8",
)


## 9. Funções estatísticas pareadas

In [ ]:

def bootstrap_mean_ci(diff, n_boot=N_BOOTSTRAP, seed=RANDOM_SEED):
    values = np.asarray(diff, dtype=float)
    values = values[np.isfinite(values)]
    rng = np.random.default_rng(seed)

    samples = rng.choice(values, size=(n_boot, len(values)), replace=True)
    boot_means = samples.mean(axis=1)

    return {
        "ci_low": float(np.percentile(boot_means, 2.5)),
        "ci_high": float(np.percentile(boot_means, 97.5)),
        "bootstrap_superiority": float(np.mean(boot_means > 0)),
    }


def paired_permutation_pvalue(diff, n_perm=N_PERMUTATIONS, seed=RANDOM_SEED):
    values = np.asarray(diff, dtype=float)
    values = values[np.isfinite(values)]

    observed = abs(values.mean())
    rng = np.random.default_rng(seed)
    signs = rng.choice([-1.0, 1.0], size=(n_perm, len(values)))
    permuted = np.abs((signs * values).mean(axis=1))

    return float((np.sum(permuted >= observed) + 1) / (n_perm + 1))


def rank_biserial_paired(diff):
    values = np.asarray(diff, dtype=float)
    values = values[np.isfinite(values)]
    values = values[np.abs(values) > TIE_TOLERANCE]

    if len(values) == 0:
        return 0.0

    ranks = pd.Series(np.abs(values)).rank(method="average").to_numpy()
    positive = ranks[values > 0].sum()
    negative = ranks[values < 0].sum()
    denominator = positive + negative

    return float((positive - negative) / denominator) if denominator else 0.0


def wilcoxon_safe(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    diff = a - b

    if np.all(np.abs(diff) <= TIE_TOLERANCE):
        return 0.0, 1.0

    try:
        stat, p = wilcoxon(
            a,
            b,
            alternative="two-sided",
            zero_method="wilcox",
            method="auto",
        )
        return float(stat), float(p)
    except ValueError:
        return np.nan, 1.0


def summarize_paired(group, metric, label):
    rag_col = f"{metric}_rag"
    no_col = f"{metric}_no_rag"

    a = group[rag_col].astype(float).to_numpy()
    b = group[no_col].astype(float).to_numpy()
    diff = a - b

    w_stat, w_p = wilcoxon_safe(a, b)
    bootstrap = bootstrap_mean_ci(diff)
    permutation_p = paired_permutation_pvalue(diff)

    wins = int(np.sum(diff > TIE_TOLERANCE))
    losses = int(np.sum(diff < -TIE_TOLERANCE))
    ties = int(len(diff) - wins - losses)

    return {
        "group": label,
        "metric": metric,
        "n_pairs": len(group),
        "rag_mean": float(np.mean(a)),
        "rag_std": float(np.std(a, ddof=1)) if len(a) > 1 else 0.0,
        "no_rag_mean": float(np.mean(b)),
        "no_rag_std": float(np.std(b, ddof=1)) if len(b) > 1 else 0.0,
        "mean_difference": float(np.mean(diff)),
        "median_difference": float(np.median(diff)),
        "relative_gain_percent": (
            float(100 * np.mean(diff) / np.mean(b))
            if abs(np.mean(b)) > TIE_TOLERANCE else np.nan
        ),
        "ci95_low": bootstrap["ci_low"],
        "ci95_high": bootstrap["ci_high"],
        "bootstrap_superiority": bootstrap["bootstrap_superiority"],
        "wilcoxon_statistic": w_stat,
        "wilcoxon_p": w_p,
        "permutation_p": permutation_p,
        "rank_biserial": rank_biserial_paired(diff),
        "wins": wins,
        "ties": ties,
        "losses": losses,
    }


METRICS = ["cosine", "rougeL", "bertscore"]


## 10. Resultados globais, por modelo e por classe

In [ ]:

global_rows = [
    summarize_paired(paired, metric, "global")
    for metric in METRICS
]
global_analysis = pd.DataFrame(global_rows)

# Holm aplicado aos três testes globais de cada família
global_analysis["wilcoxon_p_holm"] = multipletests(
    global_analysis["wilcoxon_p"].fillna(1.0),
    method="holm",
)[1]
global_analysis["permutation_p_holm"] = multipletests(
    global_analysis["permutation_p"].fillna(1.0),
    method="holm",
)[1]

model_rows = []
for model_key, group in paired.groupby("model_key", sort=True):
    for metric in METRICS:
        model_rows.append(summarize_paired(group, metric, model_key))

analysis_by_model = pd.DataFrame(model_rows)
analysis_by_model["wilcoxon_p_holm_within_metric"] = np.nan
analysis_by_model["permutation_p_holm_within_metric"] = np.nan

for metric in METRICS:
    idx = analysis_by_model["metric"].eq(metric)
    analysis_by_model.loc[idx, "wilcoxon_p_holm_within_metric"] = multipletests(
        analysis_by_model.loc[idx, "wilcoxon_p"].fillna(1.0),
        method="holm",
    )[1]
    analysis_by_model.loc[idx, "permutation_p_holm_within_metric"] = multipletests(
        analysis_by_model.loc[idx, "permutation_p"].fillna(1.0),
        method="holm",
    )[1]

class_rows = []
for class_name, group in paired.groupby("canonical_class", sort=True):
    for metric in METRICS:
        class_rows.append(summarize_paired(group, metric, class_name))

analysis_by_class = pd.DataFrame(class_rows)
analysis_by_class["wilcoxon_p_holm_within_metric"] = np.nan
analysis_by_class["permutation_p_holm_within_metric"] = np.nan

for metric in METRICS:
    idx = analysis_by_class["metric"].eq(metric)
    analysis_by_class.loc[idx, "wilcoxon_p_holm_within_metric"] = multipletests(
        analysis_by_class.loc[idx, "wilcoxon_p"].fillna(1.0),
        method="holm",
    )[1]
    analysis_by_class.loc[idx, "permutation_p_holm_within_metric"] = multipletests(
        analysis_by_class.loc[idx, "permutation_p"].fillna(1.0),
        method="holm",
    )[1]

global_analysis.to_csv(
    TABLE_DIR / "robust_global_analysis.csv",
    index=False,
    encoding="utf-8",
)
analysis_by_model.to_csv(
    TABLE_DIR / "robust_analysis_by_model.csv",
    index=False,
    encoding="utf-8",
)
analysis_by_class.to_csv(
    TABLE_DIR / "robust_analysis_by_class.csv",
    index=False,
    encoding="utf-8",
)

print("ANÁLISE GLOBAL")
display(global_analysis)

print("ANÁLISE POR MODELO")
display(analysis_by_model)

print("ANÁLISE POR CLASSE")
display(analysis_by_class)


## 11. Análise de sensibilidade — somente pares válidos nas duas condições

In [ ]:

valid_pairs = paired[
    paired["valid_report_rag"].astype(bool)
    & paired["valid_report_no_rag"].astype(bool)
].copy()

sensitivity_rows = []

if len(valid_pairs) >= 2:
    for metric in METRICS:
        sensitivity_rows.append(
            summarize_paired(valid_pairs, metric, "both_reports_valid")
        )

sensitivity_analysis = pd.DataFrame(sensitivity_rows)

if not sensitivity_analysis.empty:
    sensitivity_analysis["wilcoxon_p_holm"] = multipletests(
        sensitivity_analysis["wilcoxon_p"].fillna(1.0),
        method="holm",
    )[1]
    sensitivity_analysis["permutation_p_holm"] = multipletests(
        sensitivity_analysis["permutation_p"].fillna(1.0),
        method="holm",
    )[1]

sensitivity_analysis.to_csv(
    TABLE_DIR / "sensitivity_analysis_valid_pairs.csv",
    index=False,
    encoding="utf-8",
)

print(f"Pares válidos nas duas condições: {len(valid_pairs)} de {len(paired)}")
display(sensitivity_analysis)


## 12. Tabelas compactas para o artigo

In [ ]:

article_global = global_analysis[
    [
        "metric",
        "n_pairs",
        "rag_mean",
        "rag_std",
        "no_rag_mean",
        "no_rag_std",
        "mean_difference",
        "relative_gain_percent",
        "ci95_low",
        "ci95_high",
        "wilcoxon_p",
        "wilcoxon_p_holm",
        "rank_biserial",
        "wins",
        "ties",
        "losses",
    ]
].copy()

article_by_model = analysis_by_model[
    [
        "group",
        "metric",
        "n_pairs",
        "rag_mean",
        "no_rag_mean",
        "mean_difference",
        "relative_gain_percent",
        "wins",
        "ties",
        "losses",
    ]
].copy()

article_by_class = analysis_by_class[
    [
        "group",
        "metric",
        "n_pairs",
        "rag_mean",
        "no_rag_mean",
        "mean_difference",
        "wins",
        "ties",
        "losses",
    ]
].copy()

article_global.to_csv(
    TABLE_DIR / "article_global_table.csv",
    index=False,
    encoding="utf-8",
)
article_by_model.to_csv(
    TABLE_DIR / "article_by_model_table.csv",
    index=False,
    encoding="utf-8",
)
article_by_class.to_csv(
    TABLE_DIR / "article_by_class_table.csv",
    index=False,
    encoding="utf-8",
)

with pd.ExcelWriter(TABLE_DIR / "ablation_tables.xlsx", engine="openpyxl") as writer:
    validation_df.to_excel(writer, sheet_name="Validation", index=False)
    paired[metric_cols].to_excel(writer, sheet_name="Case Metrics", index=False)
    global_analysis.to_excel(writer, sheet_name="Global", index=False)
    analysis_by_model.to_excel(writer, sheet_name="By Model", index=False)
    analysis_by_class.to_excel(writer, sheet_name="By Class", index=False)
    sensitivity_analysis.to_excel(writer, sheet_name="Sensitivity", index=False)
    reference_df.to_excel(writer, sheet_name="References", index=False)

display(article_global)


## 13. Casos em que o RAG ganhou, empatou ou perdeu

In [ ]:

case_level = paired[
    [
        "pair_id",
        "model_key",
        "sample_id",
        "canonical_class",
        "valid_report_rag",
        "valid_report_no_rag",
        "report_rag",
        "report_no_rag",
        "reference_report",
        "cosine_rag",
        "cosine_no_rag",
        "cosine_delta",
        "rougeL_rag",
        "rougeL_no_rag",
        "rougeL_delta",
        "bertscore_rag",
        "bertscore_no_rag",
        "bertscore_delta",
        "rag_vs_no_rag_cosine",
    ]
].copy()

for metric in METRICS:
    delta = case_level[f"{metric}_delta"]
    case_level[f"{metric}_outcome"] = np.select(
        [
            delta > TIE_TOLERANCE,
            delta < -TIE_TOLERANCE,
        ],
        [
            "RAG_wins",
            "No_RAG_wins",
        ],
        default="Tie",
    )

case_level.to_csv(
    TABLE_DIR / "case_level_differences.csv",
    index=False,
    encoding="utf-8",
)

display(
    case_level[
        [
            "model_key",
            "canonical_class",
            "cosine_delta",
            "rougeL_delta",
            "bertscore_delta",
            "cosine_outcome",
            "rougeL_outcome",
            "bertscore_outcome",
        ]
    ].sort_values(["model_key", "canonical_class"])
)


## 14. Gráficos pareados

In [ ]:

METRIC_LABELS = {
    "cosine": "Cosine Similarity",
    "rougeL": "ROUGE-L F1",
    "bertscore": "BERTScore F1",
}

for metric in METRICS:
    plt.figure(figsize=(7.2, 5.0))

    x = np.array([0, 1])
    for _, row in paired.iterrows():
        y = [row[f"{metric}_no_rag"], row[f"{metric}_rag"]]
        plt.plot(x, y, marker="o", alpha=0.45)

    means = [
        paired[f"{metric}_no_rag"].mean(),
        paired[f"{metric}_rag"].mean(),
    ]
    plt.plot(x, means, marker="o", linewidth=3, label="Mean")

    plt.xticks(x, ["Without RAG", "With RAG"])
    plt.ylabel(METRIC_LABELS[metric])
    plt.title(f"Paired comparison — {METRIC_LABELS[metric]}")
    plt.grid(axis="y", alpha=0.25)
    plt.legend()
    plt.tight_layout()

    path = FIG_DIR / f"paired_{metric}.png"
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.show()


## 15. Gráficos das diferenças por modelo

In [ ]:

for metric in METRICS:
    plot_df = (
        paired.groupby("model_key", as_index=False)[f"{metric}_delta"]
        .mean()
        .sort_values("model_key")
    )

    plt.figure(figsize=(7.2, 4.8))
    plt.bar(plot_df["model_key"], plot_df[f"{metric}_delta"])
    plt.axhline(0, linewidth=1)
    plt.ylabel(f"Mean Δ ({METRIC_LABELS[metric]})")
    plt.xlabel("Model")
    plt.title(f"RAG − Without RAG by model — {METRIC_LABELS[metric]}")
    plt.grid(axis="y", alpha=0.25)
    plt.tight_layout()

    path = FIG_DIR / f"delta_by_model_{metric}.png"
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.show()


## 16. Resumo operacional

In [ ]:

operational_rows = []

for condition_name, source_df in [
    ("with_rag", rag),
    ("without_rag", no_rag),
]:
    for model_key, group in source_df.groupby("model_key", sort=True):
        row = {
            "condition": condition_name,
            "model_key": model_key,
            "n_reports": len(group),
            "valid_report_rate": group["valid_report"].mean(),
        }

        for col in [
            "latency_seconds",
            "input_tokens",
            "output_tokens",
            "peak_vram_gb",
            "was_truncated",
        ]:
            if col in group.columns:
                numeric = pd.to_numeric(group[col], errors="coerce")
                if col == "peak_vram_gb":
                    row[col] = numeric.max()
                elif col == "was_truncated":
                    row["truncation_rate"] = numeric.mean()
                else:
                    row[f"{col}_mean"] = numeric.mean()

        operational_rows.append(row)

operational_summary = pd.DataFrame(operational_rows)
operational_summary.to_csv(
    TABLE_DIR / "operational_summary.csv",
    index=False,
    encoding="utf-8",
)

display(operational_summary)


## 17. Manifesto de reprodutibilidade

In [ ]:

def sha256_file(path: Path):
    digest = hashlib.sha256()
    with open(path, "rb") as file_obj:
        for block in iter(lambda: file_obj.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


manifest = {
    "analysis_name": "RAG_vs_without_RAG_ablation",
    "analysis_version": "v1",
    "expected_pairs": EXPECTED_PAIRS,
    "actual_pairs": len(paired),
    "models": sorted(paired["model_key"].unique().tolist()),
    "classes": sorted(paired["canonical_class"].unique().tolist()),
    "pairing_key": ["model_key", "sample_id"],
    "metrics": {
        "cosine": "sentence-transformers/all-mpnet-base-v2",
        "rougeL": "rouge-score RougeScorer with English stemmer",
        "bertscore": "microsoft/deberta-xlarge-mnli",
    },
    "statistics": {
        "wilcoxon": "two-sided paired Wilcoxon signed-rank",
        "bootstrap_iterations": N_BOOTSTRAP,
        "permutation_iterations": N_PERMUTATIONS,
        "multiple_testing": "Holm",
        "random_seed": RANDOM_SEED,
    },
    "main_analysis_policy": "all 24 pairs retained",
    "sensitivity_policy": "pairs valid in both conditions reported separately",
    "input_files": {
        rag_zip.name: sha256_file(rag_zip),
        no_rag_zip.name: sha256_file(no_rag_zip),
    },
}

with open(OUTPUT_DIR / "analysis_manifest.json", "w", encoding="utf-8") as file_obj:
    json.dump(manifest, file_obj, ensure_ascii=False, indent=2)

print(json.dumps(manifest, ensure_ascii=False, indent=2))


## 18. Compactação e download dos resultados

In [ ]:

ZIP_PATH = Path("/content/ablation_analysis_final.zip")

if ZIP_PATH.exists():
    ZIP_PATH.unlink()

shutil.make_archive(
    str(ZIP_PATH.with_suffix("")),
    "zip",
    root_dir=OUTPUT_DIR,
)

print("Arquivo final:", ZIP_PATH)
print(f"Tamanho: {ZIP_PATH.stat().st_size / 1024:.1f} KB")

files.download(str(ZIP_PATH))
